# pandas 1/3 : Lire et explorer des données

**Objectifs** : charger un fichier CSV avec `read_csv`, comprendre `Series` et `DataFrame`, explorer un jeu de données et créer de nouvelles colonnes.

Nous travaillerons sur le célèbre jeu de données du **Titanic** (891 passagers).

*Durée estimée : 40 min.*

In [ ]:
# Cellule de préparation : à exécuter en premier (ne fait rien en local)
import os, sys
if "google.colab" in sys.modules and not os.path.exists("intro-data-python"):
    !git clone -q https://github.com/cdrutinuspro/intro-data-python.git
    %cd intro-data-python/notebooks

## 1. Series et DataFrame

- Une **Series** est une colonne de données (une array NumPy avec un *index* d'étiquettes).
- Un **DataFrame** est un tableau : un ensemble de Series partageant le même index.

In [ ]:
import pandas as pd
import numpy as np

s = pd.Series([12, 15, 9], index=["Alice", "Bob", "Chloé"])
print(s)
print()
print(s["Bob"])
print(s.mean())

In [ ]:
df_petit = pd.DataFrame({
    "nom": ["Alice", "Bob", "Chloé"],
    "age": [20, 22, 21],
    "note": [12.5, 15.0, 9.0],
})
df_petit

## 2. Lire un fichier CSV : `read_csv`

Un fichier CSV est un fichier texte où les valeurs sont séparées par un caractère (souvent `,`). Voici les colonnes du fichier `titanic.csv` :

| Colonne | Signification |
|---|---|
| `PassengerId` | identifiant du passager |
| `Survived` | 1 = survivant, 0 = décédé |
| `Pclass` | classe du billet (1, 2 ou 3) |
| `Name`, `Sex`, `Age` | identité |
| `SibSp` | nombre de frères/sœurs/conjoints à bord |
| `Parch` | nombre de parents/enfants à bord |
| `Ticket`, `Fare`, `Cabin` | billet, tarif, cabine |
| `Embarked` | port d'embarquement (C = Cherbourg, Q = Queenstown, S = Southampton) |

In [ ]:
df = pd.read_csv("../data/titanic.csv")
df.head()

`head()` affiche les 5 premières lignes, `tail()` les 5 dernières, et `head(10)` les 10 premières.

In [ ]:
df.tail(3)

## 3. Première exploration

In [ ]:
print(df.shape)       # (lignes, colonnes)
print(df.columns)
print(df.dtypes)

In [ ]:
df.info()

`info()` est très utile : il donne le nombre de valeurs **non nulles** par colonne. On voit déjà que `Age` et `Cabin` ont des valeurs manquantes.

In [ ]:
df.describe()   # statistiques sur les colonnes numériques

In [ ]:
df.describe(include="all")   # toutes les colonnes

### Valeurs manquantes

Les valeurs manquantes sont représentées par `NaN`. `isna()` renvoie des booléens, et `sum()` les compte.

In [ ]:
df.isna().sum()

### Compter les valeurs : `value_counts`

In [ ]:
print(df["Sex"].value_counts())
print()
print(df["Pclass"].value_counts(normalize=True))   # en proportions

## 4. Sélectionner des colonnes

- `df["Age"]` renvoie une **Series**
- `df[["Name", "Age"]]` (double crochet : on passe une liste) renvoie un **DataFrame**

In [ ]:
ages = df["Age"]
print(type(ages))
print(ages.head())

sous_df = df[["Name", "Age"]]
print(type(sous_df))
sous_df.head()

### Calculs sur une colonne

Les colonnes se comportent comme des arrays NumPy : opérations vectorisées, agrégations, etc. Les `NaN` sont **ignorés** par défaut dans les agrégations.

In [ ]:
print("Âge moyen   :", df["Age"].mean())
print("Âge max     :", df["Age"].max())
print("Tarif médian:", df["Fare"].median())
print("Survie      :", df["Survived"].mean())   # moyenne de 0/1 = proportion de 1
print("Ports       :", df["Embarked"].unique())
print("Nb de ports :", df["Embarked"].nunique())

## 5. Créer de nouvelles colonnes

Il suffit d'affecter à un nom de colonne (existant ou non).

In [ ]:
df["TailleFamille"] = df["SibSp"] + df["Parch"] + 1
df["Fare_arrondi"] = df["Fare"].round(0)
df[["Name", "SibSp", "Parch", "TailleFamille", "Fare_arrondi"]].head()

## 6. Trier : `sort_values`

In [ ]:
df.sort_values("Fare", ascending=False).head()[["Name", "Pclass", "Fare"]]

In [ ]:
# Plusieurs critères : par classe croissante, puis par âge décroissant
df.sort_values(["Pclass", "Age"], ascending=[True, False]).head()[["Name", "Pclass", "Age"]]

## 7. Options utiles de `read_csv`

| Option | Rôle |
|---|---|
| `sep=";"` | caractère séparateur |
| `decimal=","` | séparateur décimal (format français) |
| `index_col="col"` | utiliser une colonne comme index |
| `usecols=[...]` | ne lire que certaines colonnes |
| `nrows=n` | ne lire que n lignes |
| `na_values=[...]` | valeurs à considérer comme manquantes |

Un fichier CSV « à la française » (séparateur `;` et virgule décimale) :

In [ ]:
from io import StringIO

texte = "nom;taille;poids\nAlice;1,68;60,5\nBob;1,82;75\n"
print(pd.read_csv(StringIO(texte)))               # sans options : raté
print()
print(pd.read_csv(StringIO(texte), sep=";", decimal=","))   # avec options

In [ ]:
df2 = pd.read_csv("../data/titanic.csv", index_col="PassengerId", usecols=["PassengerId", "Name", "Age", "Fare"], nrows=5)
df2

### Écrire un CSV : `to_csv`

`index=False` évite d'écrire l'index comme colonne supplémentaire.

In [ ]:
df2.to_csv("extrait.csv", index=False)
print(pd.read_csv("extrait.csv"))

## Récapitulatif

| Je veux… | Code |
|---|---|
| lire un CSV | `pd.read_csv(chemin)` |
| aperçu | `df.head()`, `df.tail()` |
| dimensions, types | `df.shape`, `df.dtypes`, `df.info()` |
| statistiques | `df.describe()` |
| valeurs manquantes | `df.isna().sum()` |
| compter des modalités | `df["col"].value_counts()` |
| une / plusieurs colonnes | `df["col"]` / `df[["c1", "c2"]]` |
| nouvelle colonne | `df["nouvelle"] = ...` |
| trier | `df.sort_values("col")` |

---
## Exercices

### ✏️ Exercice 1

Lisez `titanic.csv` dans un DataFrame `df`. Combien de lignes et de colonnes contient-il ?

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
df = pd.read_csv("../data/titanic.csv")
print(df.shape)
```

</details>

### ✏️ Exercice 2

Quelle colonne a le plus de valeurs manquantes ? Combien ?

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
manquants = df.isna().sum()
print(manquants.idxmax(), manquants.max())
```

</details>

### ✏️ Exercice 3

Quel est l'âge moyen des passagers ? Quel est le tarif moyen du billet ? Quelle est la proportion d'hommes à bord ?

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
print(df["Age"].mean())
print(df["Fare"].mean())
print(df["Sex"].value_counts(normalize=True)["male"])
```

</details>

### ✏️ Exercice 4

Ajoutez une colonne `FareParPersonne` égale au tarif divisé par la taille de la famille (`SibSp + Parch + 1`). Affichez les 5 plus élevées avec le nom du passager.

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
df["FareParPersonne"] = df["Fare"] / (df["SibSp"] + df["Parch"] + 1)
print(df.sort_values("FareParPersonne", ascending=False).head()[["Name", "FareParPersonne"]])
```

</details>

### ✏️ Exercice 5

Quel est le passager le plus âgé ? (nom et âge) *Indice : `sort_values`, ou `idxmax`.*

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
print(df.sort_values("Age", ascending=False).head(1)[["Name", "Age"]])
```

</details>